# 花书 · 第一章：引言

> 你打开的是 demo / work 副本。**学习时用 work 副本**——
> 教材副本只读（rebuild 时会被覆盖），做题请运行根目录的 `./start ch01`。

## 配套资料

- 📖 花书 PDF（vault）：`30 The Colonnade/36 Library/花书.pdf`（第 1 章 Introduction）
- ⚠️ 注意：`花书拆解/重要章节/` **没有第 1 章的朱明超精读版**（从第 2 章起才有）。
  本章的「📖 观点提要」是 blossom 按原书第 1 章公开内容结构整理转述的，**不是原文引用**
- 🔧 Jupyter / NumPy 速查：忘了的话 [回 ch02 Sec 0](../ch02-linear-algebra/ch02.ipynb) 查
- 🎯 用途：本章是全书**鸟瞰图**——数学几乎没有，重点是把「深度学习在整张 AI 地图上的位置」
  和「它为什么现在才成功」这两件事讲清楚。后面每一章都会回扣本章的某个论断

## 本章导航（按原书第 1 章结构走）

| 节 | 主题 |
|---|------|
| §0 | 深度学习是什么：AI ⊃ ML ⊃ 表示学习 ⊃ DL |
| §1 | 表示的力量：笛卡尔 vs 极坐标实验（原书图 1.1） |
| §2 | 深度 = 复合函数（两种深度度量） |
| §3 | 神经网络的三次浪潮（原书 §1.2.1） |
| §4 | 与日俱增的数据量（原书 §1.2.2）+ sklearn digits 初体验 |
| §5 | 与日俱增的模型规模（原书 §1.2.3） |
| §6 | 与日俱增的精度与现实影响（原书 §1.2.4） |
| CHECKPOINT | 章末自检清单 |

## 学法

1. 每节先看 📖 **观点提要**（原书第 1 章的论点，blossom 转述）
2. 跟着直觉解释 + 例题动手填 `___`
3. 跑 cell 看 `checks.assert_*` 输出 ✅ / ❌
4. **Workshop 钩子**：遇到「📍」标记的句子，记一下名字就好——具体推导留到对应章节


In [ ]:
# 本章用到的所有库一次导入
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from utils import checks, viz

np.random.seed(0)  # 保证本章随机结果可复现

---

## §0 深度学习是什么：AI ⊃ ML ⊃ 表示学习 ⊃ DL


#### 📖 花书第 1 章 · 观点提要（blossom 转述，非原文）

1. **对计算机来说，难的和人以为的相反**：形式化任务（下棋、解方程）早就被攻克——
   IBM Deep Blue 1997 年就赢了国际象棋世界冠军；真正难的是**人凭直觉就会**的任务：
   认出照片里的猫、听懂一句口语。这些任务的「规则」我们自己都说不出来。
2. **知识库路线失败了**：1980 年代的 Cyc 等项目试图把世界知识写成手工逻辑规则，
   结果规则永远写不完、彼此打架（Cyc 曾把「用剃刀刮胡子的人」推理成矛盾——
   刮胡子时人含有电动部件，那还算不算人？）。教训：**知识必须从数据里学出来**，这就是机器学习 (Machine Learning)。
3. **表示决定成败**：经典 ML（如 logistic regression）依赖人工设计的**特征 (feature)**——
   给它「产妇是否有子宫疤痕」这种特征，它能预测剖腹产风险；但直接给它 MRI 原始像素，它什么也做不了。
4. **表示学习 (Representation Learning)**：既然特征这么关键，那让机器**连特征一起学**
   （代表：autoencoder）。难点是原始数据中的**变差因素 (factors of variation)**——
   如口音、光照、视角——纠缠在一起，浅层方法解不开。
5. **深度学习 (Deep Learning)**：用**多层嵌套的简单表示**逐层解开纠缠——
   像素 → 边缘 → 角点轮廓 → 物体部件 → 物体。深度学习是表示学习的一种，
   表示学习是机器学习的一种，机器学习是实现 AI 的一条路径。


### 直觉：一张嵌套的地图

把四个概念画成同心圆（原书图 1.4 的 Venn 图）：

$$\text{AI} \supset \text{ML} \supset \text{表示学习} \supset \text{DL}$$

每一环都有「属于本环但不属于内环」的代表：

| 环 | 代表系统 | 为什么卡在这一环 |
|---|---------|----------------|
| AI（非 ML） | 知识库 / 专家系统（如 Cyc） | 规则全部**手写**，不从数据学任何东西 |
| ML（非表示学习） | logistic regression | 从数据学「特征 → 输出」的映射，但**特征是人工设计的** |
| 表示学习（非 DL） | 浅层 autoencoder | 连特征也从数据学，但只有**一层**表示变换 |
| DL | 多层感知机 MLP | 特征逐层嵌套：简单表示 → 复杂表示 |

**判断口诀**：先问「有没有从数据学东西」（有 → 进 ML 环），再问「特征是学的还是人设计的」
（学的 → 进表示学习环），最后问「表示是一层还是多层嵌套」（多层 → 进 DL 环）。


In [ ]:
# 可视化：AI ⊃ ML ⊃ 表示学习 ⊃ DL 同心圆（原书图 1.4 的 Venn 关系）
fig, ax = plt.subplots(figsize=(7, 7))
rings = [
    (4.0, '#dbeafe', 'AI', '例：知识库 (Cyc)'),
    (3.0, '#bfdbfe', 'Machine Learning', '例：logistic regression'),
    (2.0, '#93c5fd', 'Representation Learning', '例：浅层 autoencoder'),
    (1.0, '#60a5fa', 'Deep Learning', '例：MLP'),
]
for radius, color, label, example in rings:
    circle = plt.Circle((0, -(4.0 - radius)), radius, color=color, ec='k', lw=0.8)
    ax.add_patch(circle)
    ax.text(0, -(4.0 - radius) + radius - 0.42, label,
            ha='center', fontsize=11, weight='bold')
    ax.text(0, -(4.0 - radius) + radius - 0.78, example, ha='center', fontsize=8)
ax.set_xlim(-4.6, 4.6); ax.set_ylim(-8.6, 0.6)
ax.set_aspect('equal'); ax.axis('off')
ax.set_title('AI ⊃ ML ⊃ Representation Learning ⊃ Deep Learning')
plt.show()

### ✏️ 例题 0.E1：把系统放进正确的环

下面 4 个系统，各自**最内**能进到哪一环？把答案填成字符串。

**提示**：候选答案只有四个：`'AI'` / `'ML'` / `'表示学习'` / `'DL'`。
按上面的判断口诀走三问：学不学数据 → 特征谁设计 → 表示几层。


In [ ]:
# 三问口诀编码成性质表：学不学数据 / 特征是不是学的 / 表示是不是多层
learns_from_data = {'AI': False, 'ML': True, '表示学习': True, 'DL': True}
learns_features  = {'AI': False, 'ML': False, '表示学习': True, 'DL': True}
multi_layer      = {'AI': False, 'ML': False, '表示学习': False, 'DL': True}

category = {
    'Cyc 知识库（手写逻辑规则做推理）':            '___',
    'logistic regression（用人工设计的特征）':     '___',
    '浅层 autoencoder（自动学习一层特征）':        '表示学习',   # 已给一个示范
    '多层感知机 MLP（逐层学习嵌套表示）':          '___',
}
for system, ring in category.items():
    print(f'{system}  →  {ring}')

# 断言按「口诀」验证，不直接对答案——填错哪一环，就是哪一问没走对
cyc, logreg, ae, mlp = category.values()
checks.assert_true('0.E1 Cyc：不从数据学任何东西', not learns_from_data[cyc])
checks.assert_true('0.E1 logreg：学映射但特征是人设计的',
                   learns_from_data[logreg] and not learns_features[logreg])
checks.assert_true('0.E1 autoencoder：学特征但只有一层',
                   learns_features[ae] and not multi_layer[ae])
checks.assert_true('0.E1 MLP：多层嵌套表示', multi_layer[mlp])
checks.assert_equal('0.E1 四环各用一次', sorted(category.values()),
                    sorted(['AI', 'ML', '表示学习', 'DL']))

### ✏️ 例题 0.E2：用 Python 集合验证包含链

把每一环的代表系统装进 Python `set`，用集合运算验证 $\text{DL} \subseteq \text{表示学习} \subseteq \text{ML} \subseteq \text{AI}$。

**提示**：Python 集合的「子集」判断用 `<=` 运算符（`A <= B` 表示 A ⊆ B）；
外环集合 = 内环集合 ∪ 本环独有系统，所以 AI 集合最大。


In [ ]:
DL  = {'mlp'}
REP = DL | {'shallow_autoencoder'}          # 表示学习 = DL + 浅层 autoencoder
ML  = REP | {'logistic_regression'}          # ML = 表示学习 + 经典 ML
AI  = ML | {'knowledge_base'}                # AI = ML + 知识库路线

chain_holds = DL ___ REP ___ ML ___ AI       # 填子集运算符（提示见上）
print('DL ⊆ 表示学习 ⊆ ML ⊆ AI ？', chain_holds)
print('AI 环里但不做任何学习的系统：', AI - ML)

checks.assert_true('0.E2 包含链成立', chain_holds, hint='内环集合必须是外环的子集')
checks.assert_equal('0.E2 AI 独有 = 知识库', AI - ML, {'knowledge_base'})

### ✏️ 例题 0.E3：哪些部件是「学出来的」？

原书图 1.5 把四条路线画成流程图，核心差别是**灰色阴影（从数据学习）盖住了哪些框**：

| 路线 | 输入 → ... → 输出 | 学出来的部件 |
|------|------------------|-------------|
| 规则系统 | 输入 → 手写程序 → 输出 | （无） |
| 经典 ML | 输入 → 人工特征 → **映射** → 输出 | 映射 |
| 表示学习 | 输入 → **特征** → **映射** → 输出 | 特征 + 映射 |
| 深度学习 | 输入 → **简单特征** → **多层抽象特征** → **映射** → 输出 | 全部中间层 |

把每条路线「学出来的部件数」填进下面的 dict（规则系统 = 0）。

**提示**：对照上表数「学出来的部件」一栏——深度学习按 3 算（简单特征 / 抽象特征 / 映射）。


In [ ]:
learned_parts = {
    '规则系统':  0,
    '经典 ML':   ___,
    '表示学习':  ___,
    '深度学习':  ___,
}
for route, n in learned_parts.items():
    print(f'{route}: 从数据学 {n} 类部件')

checks.assert_equal('0.E3 学习部件数递增',
                    [learned_parts[k] for k in ['规则系统', '经典 ML', '表示学习', '深度学习']],
                    [0, 1, 2, 3])
checks.assert_true('0.E3 深度学习学得最多',
                   learned_parts['深度学习'] == max(learned_parts.values()),
                   hint='DL 连中间的多层抽象特征都是学的')

---

## §1 表示的力量：同一份数据，换个坐标系难度天差地别


#### 📖 花书第 1 章 · 观点提要（blossom 转述，非原文）

原书图 1.1 给了全书第一个例子：**在散点图上用一条直线分开两类点**。

- 用**笛卡尔坐标** $(x, y)$ 表示数据 → 两类点缠成同心环，**直线怎么画都分不开**
- 换成**极坐标** $(r, \theta)$ 表示同一份数据 → 一条竖线就分开了

任务没变、数据没变，**只换了表示 (representation)**，问题就从「不可能」变成「一眼出」。
原书由此引出核心论断：**信息表示方式的好坏，对机器学习算法的成败影响巨大**——
这也是「表示学习」值得专门研究的根本理由。


### 动手复现原书图 1.1

先造一份「同心环」数据：类 0 是内圈（半径 $r < 1$），类 1 是外环（半径 $r \in [2, 3]$）。
用固定种子的 `default_rng(1)`，保证每次跑出来一样。


In [ ]:
rng = np.random.default_rng(1)      # 独立随机源，避免被别的 cell 影响
n = 120                              # 每类 120 个点

r0 = rng.uniform(0.1, 1.0, n)        # 类 0：内圈半径
r1 = rng.uniform(2.0, 3.0, n)        # 类 1：外环半径
t0 = rng.uniform(0, 2 * np.pi, n)    # 角度均匀撒一圈
t1 = rng.uniform(0, 2 * np.pi, n)

# 拼成笛卡尔坐标 (x, y)
x = np.concatenate([r0 * np.cos(t0), r1 * np.cos(t1)])
y = np.concatenate([r0 * np.sin(t0), r1 * np.sin(t1)])
label = np.concatenate([np.zeros(n, dtype=int), np.ones(n, dtype=int)])

plt.figure(figsize=(5, 5))
plt.scatter(x[label == 0], y[label == 0], s=14, label='类 0（内圈）')
plt.scatter(x[label == 1], y[label == 1], s=14, label='类 1（外环）')
plt.gca().set_aspect('equal')
plt.legend(); plt.title('笛卡尔坐标：一条直线切不开同心环')
plt.show()

### ✏️ 例题 1.E1：把数据换成极坐标表示

**任务**：从 $(x, y)$ 算出每个点的极坐标 $(r, \theta)$。

**提示**（第一次用这两个函数）：
- 半径：$r = \sqrt{x^2 + y^2}$ —— 用 `np.sqrt(...)`，平方直接写 `x**2`
- 角度：$\theta = \mathrm{atan2}(y, x)$ —— 用 `np.arctan2(y, x)`（**注意参数顺序是 y 在前**；
  它比 `np.arctan(y/x)` 靠谱，因为能分清四个象限、也不怕 x=0）


In [ ]:
r = ___                              # 半径：sqrt(x² + y²)
theta = ___                          # 角度：arctan2，y 在前

print('r 前 5 个：', np.round(r[:5], 3))
print('theta 前 5 个：', np.round(theta[:5], 3))

checks.assert_shape('1.E1 r 形状', r, (240,))
checks.assert_close('1.E1 r[0] 应等于内圈半径 r0[0]', r[0], r0[0])
checks.assert_true('1.E1 换了表示后两类在 r 轴上完全分开',
                   r[label == 0].max() < r[label == 1].min(),
                   hint='内圈 r<1、外环 r>2，理应无重叠——检查 r 的公式')

In [ ]:
# 同一份数据画在 (theta, r) 平面上——一条水平线就分开了
plt.figure(figsize=(6, 4))
plt.scatter(theta[label == 0], r[label == 0], s=14, label='类 0（内圈）')
plt.scatter(theta[label == 1], r[label == 1], s=14, label='类 1（外环）')
plt.axhline(1.5, color='k', ls='--', label='r = 1.5 一刀切')
plt.xlabel('theta'); plt.ylabel('r')
plt.legend(); plt.title('极坐标表示：同一份数据变成线性可分')
plt.show()

### ✏️ 例题 1.E2：极坐标下的一刀切分类器

**任务**：用规则「$r > 1.5$ 判为类 1」做一个零参数分类器，算它的准确率。

**提示**：
- 比较运算 `r > 1.5` 给出布尔数组，`.astype(int)` 转成 0/1 预测
- 准确率 = 预测对的比例：`(pred == label).mean()`（布尔数组的均值就是 True 的比例）


In [ ]:
pred_polar = (___).astype(int)       # 规则：r > 1.5
acc_polar = (___).mean()             # 预测对的比例
print(f'极坐标一刀切准确率：{acc_polar:.3f}')

checks.assert_close('1.E2 极坐标准确率 = 100%', acc_polar, 1.0)
checks.assert_shape('1.E2 预测形状', pred_polar, (240,))

### ✏️ 例题 1.E3：预测题——回到笛卡尔坐标，一刀切还行吗？

同样的「单变量阈值」思路用在 $x$ 上：扫遍所有可能的阈值 $c$，取「$x > c$ 判类 1」
（或反过来）能达到的**最好**准确率。

**运行前先猜**：这个最好准确率大概是多少？接近 1.0，还是 0.5 附近？

**直觉**：外环在 $x$ 轴上左右都有点、内圈夹在中间——任何一个阈值都会切错一大片。


In [ ]:
best_acc_cart = 0.0
for c in np.linspace(x.min(), x.max(), 201):
    pred = (x > c).astype(int)
    acc = max((pred == label).mean(), (1 - pred == label).mean())  # 两个方向取好的
    best_acc_cart = max(best_acc_cart, acc)
print(f'笛卡尔坐标下单变量阈值的最好准确率：{best_acc_cart:.3f}')
print(f'对比极坐标：{acc_polar:.3f}')

checks.assert_true('1.E3 笛卡尔一刀切明显更差', best_acc_cart < 0.85,
                   hint='同心环在 x 轴上重叠严重，单阈值切不开')
checks.assert_true('1.E3 极坐标碾压笛卡尔', acc_polar > best_acc_cart)

### 结论 + 两个真实版本

- **数据没变，表示一换，同一个「一刀切」从 ≈0.6 涨到 1.0**。这就是原书说的
  「表示的好坏决定任务难易」的最小可复现版本。
- 真实版本 1（原书例子）：预测剖腹产风险——给 logistic regression「有没有子宫疤痕」这类
  **人工特征**它就能工作；直接喂 MRI 原始像素就不行。人工特征 = 医生大脑里做过的「换坐标系」。
- 真实版本 2：语音识别里的口音、图像里的光照——这些**变差因素**把有用信息缠住了，
  好的表示要把它们解开。人工设计解不动时，就轮到**表示学习**登场：让机器自己找「极坐标」。

📍 **Workshop 钩子**：表示学习 ⟺ **embedding**。LLM 把每个 token 映射成一个**学出来的**向量
（embedding），就是「机器自己找好坐标系」的现代形态——Day 4 写 nanoGPT 时你会亲手建这张
embedding 表。这里记住名字就行。


---

## §2 深度 = 复合函数：简单表示层层嵌套出复杂表示


#### 📖 花书第 1 章 · 观点提要（blossom 转述，非原文）

1. **深度学习的核心策略**：复杂概念不直接学，而是**用一串简单概念嵌套出来**——
   识别一张人脸：像素 → 边缘 → 角点和轮廓 → 五官部件 → 人脸（原书图 1.2）。
2. **深度怎么度量？原书给了两种观点**：
   - **计算图深度**：把模型看成一串函数复合 $f^{(3)}(f^{(2)}(f^{(1)}(x)))$，数复合了几层
   - **概念图深度**：数「概念建立在概念之上」有几级抽象
   两种数法给出的数字可以不同，**深度没有唯一正确的定义**，也没有「多深才算深」的公认门槛。
3. 因此原书对深度学习的工作定义很朴素：**学习的模型涉及比传统方法更多的复合层次**。


### ✏️ 例题 2.E1：亲手复合三层函数

取三个再简单不过的函数：

$$f_1(x) = 3x, \qquad f_2(x) = x + 2, \qquad f_3(x) = x^2$$

**任务**：写出「深度 3」的复合 $g(x) = f_3(f_2(f_1(x)))$，验证 $g(1) = (3 \cdot 1 + 2)^2 = 25$。

**提示**：Python 里函数复合就是**从里往外套括号**：`f3(f2(f1(x)))`。


In [ ]:
f1 = lambda x: 3 * x
f2 = lambda x: x + 2
f3 = lambda x: x**2

g = lambda x: ___                    # 从里往外：f1 → f2 → f3
depth = ___                          # 这个复合的计算图深度是几？

print('g(1) =', g(1))
print('g(0) =', g(0))

checks.assert_equal('2.E1 g(1)', g(1), 25)
checks.assert_equal('2.E1 g(0)', g(0), 4)
checks.assert_equal('2.E1 计算图深度', depth, 3)

### ✏️ 例题 2.E2：预测题——复合顺序能交换吗？

**运行前先猜**：$f_1(f_2(1))$ 和 $f_2(f_1(1))$ 相等吗？各是多少？

（$f_1(x) = 3x$，$f_2(x) = x + 2$——在纸上代一遍再跑。）


In [ ]:
a = ___                              # f1(f2(1))
b_val = ___                          # f2(f1(1))
print('f1(f2(1)) =', a)
print('f2(f1(1)) =', b_val)

checks.assert_equal('2.E2 f1∘f2', a, 9)
checks.assert_equal('2.E2 f2∘f1', b_val, 5)
checks.assert_true('2.E2 复合不可交换', a != b_val,
                   hint='层的先后顺序改变结果——所以网络的「层序」有意义')

### 深度带来什么：同一个简单函数，套一层比一层「弯」

取一个只有一个「弯」的抛物线 $f(x) = 4x(1-x)$（定义在 $[0,1]$ 上）。
看它自己跟自己复合：$f$、$f \circ f$、$f \circ f \circ f$——**每复合一层，波峰数量翻倍**。
一个参数都没加，复杂度全靠「深度」长出来。这是「深 vs 宽」争论里深度一方最直观的证据。


### ✏️ 例题 2.E3：画出 f、f∘f、f∘f∘f

**任务**：补全三层复合 `f(f(f(xs)))`，并在 $[0,1]$ 上把三条曲线画在一起。

**提示**：`f` 接受整个数组 `xs`（NumPy 广播），复合还是从里往外套括号。


In [ ]:
f = lambda x: 4 * x * (1 - x)
xs = np.linspace(0, 1, 400)

y1 = f(xs)
y2 = f(f(xs))
y3 = ___                             # 深度 3 的复合

plt.figure(figsize=(7, 4))
plt.plot(xs, y1, label='f（深度 1）')
plt.plot(xs, y2, label='f∘f（深度 2）')
plt.plot(xs, y3, label='f∘f∘f（深度 3）')
plt.legend(); plt.title('复合一层，波峰翻一倍——深度免费长出复杂度')
plt.show()

checks.assert_shape('2.E3 y3 形状', y3, (400,))
checks.assert_close('2.E3 f(f(f(0.2)))', f(f(f(0.2))), 0.28901376)

📍 **Workshop 钩子**：**深度 = 复合函数 ⟺ 多层网络的 forward pass**。
Ch 6 的前馈网络就是 $f^{(L)}(\cdots f^{(2)}(f^{(1)}(x)))$ 这个式子加上「每层长什么样」的细节；
Day 4 写 nanoGPT 时，`forward()` 函数里一层层往下传的就是它。现在记住这个视角就够，
每层内部的结构（线性变换 + 非线性）留到 Ch 6。


---

## §3 神经网络的三次浪潮（原书 §1.2.1）


#### 📖 花书第 1 章 · 观点提要（blossom 转述，非原文）

深度学习不是新东西——它是**同一批思想的第三次登场**，每次换一个名字：

| 浪潮 | 年代 | 当时的名字 | 代表工作 | 为什么退潮 |
|------|------|-----------|---------|-----------|
| 第 1 次 | 1940s–1960s | **控制论 (cybernetics)** | McCulloch-Pitts 神经元 (1943)、感知机 perceptron (1958)、ADALINE (1960) | 线性模型有硬伤——**连 XOR 都学不了**（Minsky & Papert, 1969），资助随之枯竭 |
| 第 2 次 | 1980s–1990s | **联结主义 (connectionism)** | 分布式表示 (distributed representation)、反向传播 backpropagation 的推广 (1986)、LSTM (1997) | 期望被吹得太高没兑现；同期 kernel 方法与图模型表现更好 |
| 第 3 次 | 2006– | **深度学习 (deep learning)** | Hinton 的 deep belief network 用贪心逐层预训练突破深网训练 (2006) | ——（正在进行） |

另外两个原书强调的点：

1. **神经科学是灵感来源，但不是精确蓝图**——我们对大脑的了解远不足以拿它当说明书；
   现代「深度学习」这个名字也刻意超越了神经视角，强调**多层复合**这一更一般的原理。
2. 前两次浪潮的思想没有错，**缺的是数据和算力**——这正是 §4、§5 两节趋势图要讲的事。


### ✏️ 例题 3.E1：把事件归到正确的浪潮

把下面 6 个事件标上浪潮编号（1 / 2 / 3）。

**提示**：对照上表的年代和代表工作；AlexNet（2012，ImageNet 夺冠）属于第三次浪潮。


In [ ]:
wave = {
    '感知机 perceptron (1958)':                    ___,
    'Minsky & Papert 指出线性模型学不了 XOR (1969)': ___,
    '反向传播训练多层网络普及 (1986)':               2,   # 已给一个示范
    'LSTM 提出 (1997)':                            ___,
    'deep belief network 逐层预训练 (2006)':        ___,
    'AlexNet 在 ImageNet 夺冠 (2012)':              ___,
}
for event, w in wave.items():
    print(f'第 {w} 次浪潮 ← {event}')

checks.assert_equal('3.E1 perceptron', wave['感知机 perceptron (1958)'], 1)
checks.assert_equal('3.E1 XOR 批评', wave['Minsky & Papert 指出线性模型学不了 XOR (1969)'], 1)
checks.assert_equal('3.E1 backprop', wave['反向传播训练多层网络普及 (1986)'], 2)
checks.assert_equal('3.E1 LSTM', wave['LSTM 提出 (1997)'], 2)
checks.assert_equal('3.E1 DBN', wave['deep belief network 逐层预训练 (2006)'], 3)
checks.assert_equal('3.E1 AlexNet', wave['AlexNet 在 ImageNet 夺冠 (2012)'], 3)

In [ ]:
# 可视化：三次浪潮时间轴（事件年份 + 浪潮色带）
bands = [(1940, 1969, '第 1 次：cybernetics', '#fde68a'),
         (1980, 1998, '第 2 次：connectionism', '#a7f3d0'),
         (2006, 2016, '第 3 次：deep learning', '#bfdbfe')]
events = [(1943, 'McCulloch-Pitts'), (1958, 'perceptron'), (1960, 'ADALINE'),
          (1969, 'XOR 批评'), (1986, 'backprop 普及'), (1997, 'LSTM'),
          (2006, 'DBN'), (2012, 'AlexNet'), (2014, 'GoogLeNet')]

fig, ax = plt.subplots(figsize=(10, 3.2))
for lo, hi, name, color in bands:
    ax.axvspan(lo, hi, color=color, alpha=0.7)
    ax.text((lo + hi) / 2, 0.86, name, ha='center', fontsize=9, weight='bold')
for yr, name in events:
    ax.axvline(yr, color='gray', lw=0.6)
    ax.text(yr, 0.06, f'{name} ({yr})', rotation=90, fontsize=7.5, va='bottom')
ax.set_xlim(1938, 2018); ax.set_ylim(0, 1); ax.set_yticks([])
ax.set_xlabel('年份'); ax.set_title('神经网络研究的三次浪潮（原书 §1.2.1）')
plt.show()

### ✏️ 例题 3.E2：亲手验证第一次浪潮的死因——线性模型学不了 XOR

XOR 真值表：$(0,0) \to 0,\ (0,1) \to 1,\ (1,0) \to 1,\ (1,1) \to 0$。
线性分类器 = 在平面上画一条直线 $w_1 x_1 + w_2 x_2 + b = 0$，一侧判 0、一侧判 1。

**任务**：暴力扫一大批直线（角度 × 截距的网格），看线性分类器在 4 个 XOR 点上**最好**能对几个。

**提示**：
- 每条直线由 `w = (cos t, sin t)` 和截距 `c` 决定，预测 `pred = (X @ w > c)`
- 准确率还是 `(pred == y_xor).mean()`；记得两个方向都试（`pred` 和 `1 - pred`）

**运行前先猜**：最好准确率是 1.0（能学会）还是 0.75（永远错一个）？


In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])

best_acc_xor = 0.0
for t in np.linspace(0, 2 * np.pi, 181):
    w = np.array([np.cos(t), np.sin(t)])
    for c in np.linspace(-2, 2, 161):
        pred = (___).astype(int)             # 直线一侧判 1：X_xor @ w > c
        acc = max((pred == y_xor).mean(), (1 - pred == y_xor).mean())
        best_acc_xor = max(best_acc_xor, acc)
print(f'线性分类器在 XOR 上的最好准确率：{best_acc_xor}')

checks.assert_close('3.E2 线性模型学不会 XOR（最多对 3/4）', best_acc_xor, 0.75)

**读结果**：扫了 181 × 161 ≈ 2.9 万条直线，没有一条能把 4 个点全分对——**上限就是 3/4**。
（严格证明只要一句：XOR 的两类点各占对角，直线的一侧是凸集，装不下一对对角点。）

这就是 1969 年那盆冷水：**单层**线性模型有原理性天花板。解药其实就是 §2 的复合——
多套一层非线性就能表示 XOR，但当时没人知道怎么**训练**多层网络，于是第一次浪潮结束。
怎么解、怎么训，留到 Ch 6（前馈网络开篇就用 XOR 当例子）。


---

## §4 与日俱增的数据量（原书 §1.2.2）+ sklearn digits 初体验


#### 📖 花书第 1 章 · 观点提要（blossom 转述，非原文）

1. 1990s 至今，算法本身变化不大——**变的是数据量**。数字化社会让训练集从几百长到几千万。
2. 原书给的经验法则（2016 年视角，监督学习）：
   - 每类约 **5,000** 个标注样本 → 深度学习可以达到可接受性能
   - 总量约 **1,000 万** 标注样本 → 可以达到或超过人类水平
3. 更小的数据集上如何发挥深度学习，是重要研究前沿（无监督/半监督利用未标注数据——名字先记下）。


### ✏️ 例题 4.E1：复现原书图 1.8——数据集规模随年代增长

下面的年份和规模**硬编码自原书图 1.8 的公开数值（均为近似值，量级正确）**。

**任务**：规模跨了 5 个数量级，线性坐标轴会把早期数据集压成一条线——
把 y 轴改成**对数坐标**。

**提示**（第一次用）：`ax.set_yscale('log')`——参数是字符串 `'log'`。


In [ ]:
# (年份, 样本数, 名称) —— 近似值，取自原书图 1.8
datasets = [
    (1936, 1.5e2, 'Iris'),
    (1989, 2.0e3, 'T vs C'),
    (1998, 6.0e4, 'MNIST'),
    (2009, 6.0e4, 'CIFAR-10'),
    (2011, 6.0e5, 'Public SVHN'),
    (2012, 1.2e6, 'ImageNet (ILSVRC)'),
    (2014, 1.1e6, 'Sports-1M'),
    (2015, 1.0e7, 'WMT 英法平行语料'),
]
years = np.array([d[0] for d in datasets])
sizes = np.array([d[1] for d in datasets])

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.scatter(years, sizes, s=40, color='C0', zorder=3)
for yr, sz, name in datasets:
    ax.annotate(name, (yr, sz), textcoords='offset points', xytext=(6, 4), fontsize=8)
ax.set_yscale(___)                   # ← 换对数坐标（字符串）
ax.axhline(5e3, color='gray', ls=':', label='5e3/类 可接受（经验法则）')
ax.axhline(1e7, color='gray', ls='--', label='1e7 达到/超过人类（经验法则）')
ax.set_xlabel('年份'); ax.set_ylabel('数据集规模（样本数，log）')
ax.set_title('数据集规模随年代增长（近似复现原书图 1.8）')
ax.legend(loc='upper left')
plt.show()

### ✏️ 例题 4.E2：从 Iris 到 WMT 跨了几个数量级？

**任务**：用 $\log_{10}$ 算 Iris (150) 到 WMT ($10^7$) 的数量级差。

**提示**（第一次用）：`np.log10(a / b)` 或 `np.log10(a) - np.log10(b)` 都行。


In [ ]:
orders = ___                         # log10(WMT / Iris)
print(f'Iris → WMT 跨了 {orders:.2f} 个数量级（约 80 年）')

checks.assert_close('4.E2 数量级差', orders, 4.824, tol=0.01)
checks.assert_true('4.E2 超过 4 个数量级', orders > 4)

### sklearn digits 初体验：亲手摸一个真实数据集

光看趋势图不过瘾——现在加载一个**真的**数据集：`sklearn` 自带的 **digits**
（1,797 张 8×8 灰度手写数字，0–9 十类；它是 MNIST 的迷你表亲，来自 UCI 手写数字库，
**不是** MNIST 本尊）。这是你在 blossom 里摸到的第一份真实数据。


In [ ]:
from sklearn.datasets import load_digits
digits = load_digits()
print('数据字段：', [k for k in dir(digits) if not k.startswith('_')][:8])
print('data  形状：', digits.data.shape)
print('images 形状：', digits.images.shape)
print('target 形状：', digits.target.shape)
print('前 10 个标签：', digits.target[:10])

### ✏️ 例题 4.E3：读懂三个 shape

`digits.data` 是 (1797, 64)，`digits.images` 是 (1797, 8, 8)——**同一批像素的两种表示**：
每张 8×8 图既可以摊平成 64 个数的一行，也可以保持 8×8 网格。

**任务**：填出样本数和每张图的像素数，并验证「第 0 张图摊平 == data 第 0 行」。

**提示**：
- `.shape` 是元组：`digits.data.shape[0]` 是行数（样本数），`[1]` 是列数（特征数）
- 把 8×8 摊平成 64：`digits.images[0].reshape(-1)`（`-1` = 让 NumPy 自己算长度）


In [ ]:
n_samples = ___                      # data.shape 的第一个数
n_pixels = ___                       # data.shape 的第二个数
flat0 = digits.images[0].___         # 把 8×8 摊平成 (64,)

print(f'{n_samples} 张图，每张 {n_pixels} 个像素')

checks.assert_equal('4.E3 样本数', n_samples, 1797)
checks.assert_equal('4.E3 每张像素数', n_pixels, 64)
checks.assert_shape('4.E3 摊平后的形状', flat0, (64,))
checks.assert_true('4.E3 摊平 == data 第 0 行', np.allclose(flat0, digits.data[0]),
                   hint='images[0] 摊平应与 data[0] 逐元素相同')

In [ ]:
# 画前 10 张手写数字看看长什么样
fig, axes = plt.subplots(2, 5, figsize=(8, 3.6))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='gray_r')
    ax.set_title(f'标签 {digits.target[i]}', fontsize=9)
    ax.axis('off')
plt.suptitle('sklearn digits：8×8 手写数字（前 10 张）')
plt.show()

### ✏️ 例题 4.E4：预测题——像素值的取值范围是多少？

**运行前先猜**：灰度像素常见范围是 0–255，digits 的像素范围也是吗？

**提示**：数组的最小/最大值用 `.min()` / `.max()`。


In [ ]:
px_min = ___                         # 全体像素的最小值
px_max = ___                         # 全体像素的最大值
print(f'像素范围：[{px_min}, {px_max}]')

checks.assert_close('4.E4 最小像素值', px_min, 0.0)
checks.assert_close('4.E4 最大像素值', px_max, 16.0)

### ✏️ 例题 4.E5：digits 够不够喂深度学习？（用 4.E1 的经验法则算）

**任务**：数一数每类有多少张图，对照「每类 5,000」的经验法则。

**提示**（第一次用）：`np.bincount(digits.target)` 统计 0–9 每个标签出现的次数。


In [ ]:
counts = ___                         # 每个标签出现次数（np.bincount）
per_class_avg = counts.mean()
print('每类张数：', counts)
print(f'平均每类 {per_class_avg:.1f} 张  vs  经验法则 5000 张/类')

checks.assert_equal('4.E5 共 10 类', len(counts), 10)
checks.assert_equal('4.E5 总数守恒', counts.sum(), 1797)
checks.assert_true('4.E5 远低于 5000/类', per_class_avg < 5000,
                   hint='结论：按原书经验法则，这种小数据集上传统 ML 往往就够')

### 埋个钩子：这批像素就是 Ch 2 的主角

注意刚才那句「**同一批像素的两种表示**」：

- 一张 8×8 图摊平 → 一个 **64 维向量**（$\boldsymbol{x} \in \mathbb{R}^{64}$）
- 整个数据集 → 一个 **1797 × 64 矩阵**（一行 = 一张图）

「图像 = 向量、数据集 = 矩阵」是深度学习最底层的世界观，而向量和矩阵的全部运算规则
就是 [Ch 2 线性代数](../ch02-linear-algebra/ch02.ipynb) 的内容——你已经（或将要）在那里
系统学过。本章只需要记住：**数据进模型前，都先变成这种矩形的数**。


---

## §5 与日俱增的模型规模（原书 §1.2.3）


#### 📖 花书第 1 章 · 观点提要（blossom 转述，非原文）

1. 联结主义的核心洞见：**大量简单单元连在一起可以表现出智能**。模型规模有两个独立的量：
   - **每个神经元的连接数**（原书图 1.10）：人工网络已达 $\sim 10^4$，**跟哺乳动物同量级**
     （受硬件而非原理限制）
   - **神经元总数**（原书图 1.11）：自 1950s 起**大约每 2.4 年翻一倍**，动力是更快的机器、
     更大的内存、更大的数据
2. 但截至原书写作（2016），最大的人工网络神经元总数也只跟**青蛙**差不多——
   按翻倍趋势外推，要到 **2050 年代**才追上人脑神经元数量。
3. 生物神经元的功能也比人工神经元复杂得多——所以「追上数量」远不等于「追上智能」。


In [ ]:
# 近似复现原书图 1.10 + 图 1.11（数值硬编码，均为近似值）
fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4.5))

# 左：每神经元连接数（原书图 1.10，近似）
conn = [(1960, 3, 'ADALINE'), (1980, 1e2, 'Neocognitron'),
        (2006, 1e3, 'GPU CNN'), (2012, 2e3, 'Multi-GPU CNN'),
        (2014, 5e3, 'GoogLeNet')]
bio_conn = [(1e4, '人 / 猫'), (1e3, '小鼠'), (1e2, '果蝇')]
axL.scatter([c[0] for c in conn], [c[1] for c in conn], s=40, color='C0', zorder=3)
for yr, v, name in conn:
    axL.annotate(name, (yr, v), textcoords='offset points', xytext=(5, 4), fontsize=8)
for v, name in bio_conn:
    axL.axhline(v, color='gray', ls=':', lw=0.8)
    axL.text(1962, v * 1.15, name, fontsize=8, color='gray')
axL.set_yscale('log'); axL.set_xlabel('年份'); axL.set_ylabel('每神经元连接数（log）')
axL.set_title('连接数：已达哺乳动物量级（近似原书图 1.10）')

# 右：神经元总数（原书图 1.11，近似）
neurons = [(1958, 1, 'perceptron'), (1960, 1, 'ADALINE'),
           (1980, 1e3, 'Neocognitron'), (1986, 1e2, '早期 backprop 网络'),
           (1998, 1e3, 'LeNet-5'), (2006, 1e5, 'DBN'),
           (2012, 1e6, 'Multi-GPU CNN'), (2014, 1e7, 'GoogLeNet')]
bio_neu = [(3.02e2, '线虫'), (1e6, '蜜蜂'), (1.6e7, '青蛙'), (8.6e10, '人')]
axR.scatter([n[0] for n in neurons], [n[1] for n in neurons], s=40, color='C1', zorder=3)
for yr, v, name in neurons:
    axR.annotate(name, (yr, v), textcoords='offset points', xytext=(5, 4), fontsize=8)
for v, name in bio_neu:
    axR.axhline(v, color='gray', ls=':', lw=0.8)
    axR.text(1960, v * 1.6, name, fontsize=8, color='gray')
axR.set_yscale('log'); axR.set_xlabel('年份'); axR.set_ylabel('神经元总数（log）')
axR.set_title('神经元数：2016 年还只是青蛙水平（近似原书图 1.11）')
plt.tight_layout(); plt.show()

### ✏️ 例题 5.E1：外推「2050 年代」是怎么算出来的

设 2014 年最大网络约 $10^6$ 个神经元，人脑约 $8.6 \times 10^{10}$ 个，每 **2.4 年**翻一倍。

**任务**：算需要翻倍多少次、多少年，验证原书「2050 年代」的说法。

**提示**（第一次用）：翻倍次数 = $\log_2(\text{目标}/\text{起点})$ —— 用 `np.log2(...)`；
年数 = 翻倍次数 × 2.4。


In [ ]:
n_doublings = ___                    # log2(人脑 / 2014 年模型)
years_needed = ___                   # 每次翻倍 2.4 年
arrival = 2014 + years_needed
print(f'需要翻倍 {n_doublings:.1f} 次 ≈ {years_needed:.1f} 年 → 约 {arrival:.0f} 年')

checks.assert_close('5.E1 翻倍次数', n_doublings, 16.39, tol=0.05)
checks.assert_close('5.E1 所需年数', years_needed, 39.3, tol=0.2)
checks.assert_true('5.E1 落在 2050 年代', 2050 <= arrival < 2060,
                   hint='原书结论：除非有新技术，2050s 之前达不到人脑神经元数')

### 两个必须记住的「但是」

1. **数量 ≠ 智能**：生物神经元本身比人工神经元复杂得多，大脑的接线方式我们也远没搞清——
   即使哪天神经元数追平，也不自动等于人类智能。
2. **外推是危险的**：「每 2.4 年翻一倍」只是把过去的直线往前画。花书成书于 2016；
   之后 Transformer（2017）和 LLM 把「模型规模」的主角从神经元数换成了**参数量**，
   增长曲线也被改写。📍 **Workshop 钩子**：这条「规模 → 能力」曲线的现代版叫
   **scaling laws**——GPT 系列就是沿着它长出来的，Day 3 讲 LLM 训练时会正面遇到。
   这里记住名字就好。


---

## §6 与日俱增的精度与现实影响（原书 §1.2.4）


#### 📖 花书第 1 章 · 观点提要（blossom 转述，非原文）

1. 深度学习不只是模型变大，**精度也在逐年碾压**：标志性事件是 ILSVRC 图像识别竞赛——
   2012 年 AlexNet 把 top-5 错误率从 26.1% 一举砍到 15.3%，此后逐年下降到 3.6%（2015）。
2. 同样的故事发生在语音识别（错误率断崖式下降）、行人检测、图像分割、机器翻译上。
3. 应用版图随之扩张：制药、物理、围棋（当时 AlphaGo 尚在酝酿）、以及**深度强化学习**
   （deep reinforcement learning——只记名字，本书不展开）。
4. 深度学习同时反哺其他学科：神经科学用它建模、公司用它服务亿级用户。


### ✏️ 例题 6.E1：画 ILSVRC top-5 错误率的下降曲线

下面是 ILSVRC 历年冠军 top-5 错误率（**近似值**：2010 NEC、2011 XRCE、2012 AlexNet、
2013 Clarifai、2014 GoogLeNet、2015 ResNet）。

**任务**：算 2010 → 2015 错误率的**相对降幅**（降了原来的百分之几）。

**提示**：相对降幅 = (起点 − 终点) / 起点。


In [ ]:
ilsvrc_years = np.array([2010, 2011, 2012, 2013, 2014, 2015])
top5_err = np.array([0.281, 0.257, 0.153, 0.117, 0.067, 0.036])   # 近似值

rel_drop = ___                       # (起点 − 终点) / 起点
print(f'2010 → 2015 相对降幅：{rel_drop:.1%}')

plt.figure(figsize=(7, 4))
plt.plot(ilsvrc_years, top5_err * 100, 'o-', color='C3')
plt.annotate('AlexNet：深度学习进场', (2012, 15.3), textcoords='offset points',
             xytext=(10, 10), fontsize=9)
plt.axhline(5.1, color='gray', ls=':', label='人类水平 ≈ 5.1%（近似）')
plt.xlabel('年份'); plt.ylabel('top-5 错误率 (%)')
plt.title('ILSVRC 冠军错误率逐年下降（近似值）')
plt.legend(); plt.show()

checks.assert_close('6.E1 相对降幅', rel_drop, 0.872, tol=0.005)
checks.assert_true('6.E1 错误率逐年单调下降', bool(np.all(np.diff(top5_err) < 0)))

### ✏️ 例题 6.E2：预测题——哪一年降幅最大？

**运行前先猜**：上面曲线里哪一年错误率**降得最狠**？

**提示**（第一次用）：`np.diff(a)` 给相邻差值（长度少 1）；`np.argmin(d)` 给最小值
（= 最大跌幅）的下标。注意 `diff` 的第 $i$ 个差对应 `ilsvrc_years[i+1]` 那年。


In [ ]:
drops = np.diff(top5_err)
year_biggest = ilsvrc_years[___ + 1]     # 最大跌幅的下标（np.argmin）
print('降幅最大的一年：', year_biggest)

checks.assert_equal('6.E2 拐点 = AlexNet 那年', int(year_biggest), 2012)

### 本章收束：全书路线图

把第一章的五个论断串成一句话：

> 直觉类任务的知识写不出来（§0）→ 必须学，而**表示**决定学的难易（§1）→
> 深度 = 用复合函数逐层造出好表示（§2）→ 思想早就有（§3），
> 等到**数据**（§4）和**算力/规模**（§5）跟上才爆发 → 于是精度和影响一路上行（§6）。

接下来的路线（原书第一部分 = 数学工具箱）：

- [Ch 2 线性代数](../ch02-linear-algebra/ch02.ipynb)：数据和模型的语言（向量/矩阵/张量）
- [Ch 3 概率与信息论](../ch03-probability/ch03.ipynb)：不确定性的语言（分布/熵/KL）
- Ch 4 数值计算：让数学在有限精度的机器上不翻车（待建）
- Ch 5 机器学习基础 → Ch 6 起进入深度网络本体


---

## CHECKPOINT — 本章自检

在下面打勾（双击 cell 进编辑模式，把 `[ ]` 改成 `[x]`）：

**核心（§0–§2）**

- [ ] 能画出 AI ⊃ ML ⊃ 表示学习 ⊃ DL 的同心圆，并给每一环举一个代表系统
- [ ] 能用「三问口诀」（学不学数据 / 特征谁设计 / 表示几层）给任意系统定位
- [ ] 能复述知识库路线（Cyc）失败的原因
- [ ] 能用同心环 → 极坐标的实验说明「表示的好坏决定任务难易」
- [ ] 知道深度的两种度量（计算图深度 vs 概念图深度），且深度没有唯一定义
- [ ] 能手写三层函数复合 `f3(f2(f1(x)))` 并说出「复合不可交换」

**历史与趋势（§3–§6）**

- [ ] 能按顺序说出三次浪潮的名字、年代、代表工作和退潮原因
- [ ] 知道线性模型学不了 XOR，以及这件事杀死了第一次浪潮
- [ ] 记得两条数据量经验法则（5,000/类 可接受；10⁷ 达到/超过人类）
- [ ] 知道「神经元数每 2.4 年翻倍 → 2050s 才到人脑量级」的估算怎么算
- [ ] 知道 2012 年 AlexNet 是 ILSVRC 错误率曲线的拐点

**轻量预告（记住名字就行）**

- [ ] embedding（= 学出来的表示，Day 4 nanoGPT 亲手建）
- [ ] forward pass（= 多层复合函数，Ch 6）
- [ ] scaling laws（= 规模 → 能力曲线的现代版，Day 3）


In [ ]:
checks.report()